In [ ]:
# pip install -e /root/capsule

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import glob
import pickle
import os
import logging
logging.basicConfig(
    level=logging.INFO, 
    format='%(filename)s:%(lineno)d - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
SESSION_KEYS = ["subject_id", "session_date"]

import json
import requests

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'
import seaborn as sns

from aind_behavior_gym.dynamic_foraging.task import CoupledBlockTask, UncoupledBlockTask
from aind_dynamic_foraging_models.generative_model import ForagerCollection

from aind_analysis_arch_result_access.han_pipeline import get_session_table, get_mle_model_fitting
from aind_analysis_arch_result_access.util.s3 import get_s3_pkl, get_s3_json

import aind_dynamic_foraging_population_analysis

In [ ]:
model_types = [
    'QLearning_L2F1_softmax', 'QLearning_L1F1_CK1_softmax', 'ForagingCompareThreshold', 
    # 'CTTAvg'
    # 'CTTMeanReset'
]

task_types = [
    'Uncoupled Baiting', 
    'Uncoupled Without Baiting', 
    'Coupled Baiting', 
    #'None'
]

later_stages = [
    'STAGE_FINAL', 'GRADUATED'
]

## load data

In [ ]:
# load df_session_history

df_session_history = pd.read_pickle(
    os.path.expanduser("~/capsule/analysis_result/df_session_history_250714.pkl")
)
# df_session_history = pd.read_pickle(
#     os.path.expanduser("~/capsule/analysis_result/df_model_simulation-CTTDualQ-250821-uncoupled_no_baiting-0_1964.pkl")
# )

print(f"Loaded {len(df_session_history)} rows of session history data.")


# insert switch info into df_session_history
# df_session_history = insert_switch_info(df_session_history)
# print(f"Inserted switch info into df_session_history. Now has {len(df_session_history)} rows.")

df_session_history.head()

## load simulated data

In [ ]:
# load saved simulation results: hattori, bari, foraging
simulation_saved_file = os.path.expanduser("~/capsule/analysis_result/dict_simulation_results_250716.pkl")
with open(simulation_saved_file, 'rb') as f:
    dict_simulation_results = pickle.load(f)

In [ ]:
df_sim_tmp = dict_simulation_results[(model_types[0], task_types[0])]
print(f"Loaded {len(df_sim_tmp)} rows of simulation results for model {model_types[0]} and task {task_types[0]}.")
df_sim_tmp.head()

In [ ]:
# load simulation results: ctt_avg
with open("/root/capsule/analysis_result/dict_simulation_results-CTTAvg-251014.pkl", 'rb') as f:
    dict_simulation_results_ctt_avg = pickle.load(f)

In [ ]:
# # load and merge simulation results: CTTAvg, archived
# with open('/root/capsule/analysis_result/dict_simulation_results-CTTAvg-251007-coupled_baiting.pkl', 'rb') as fi:
#     dict_simulation_results_ctt_avg_coupled_baiting = pickle.load(fi)

# with open('/root/capsule/analysis_result/dict_simulation_results-CTTAvg-251007-uncoupled_baiting.pkl', 'rb') as fi:
#     dict_simulation_results_ctt_avg_uncoupled_baiting = pickle.load(fi)

# # # empty
# # with open('/root/capsule/analysis_result/dict_simulation_results-CTTAvg-251007-uncoupled_no_baiting-0_322.pkl', 'rb') as fi:
# #     dict_simulation_results_ctt_avg_uncoupled_no_baiting_0_322 = pickle.load(fi)

# with open('/root/capsule/analysis_result/dict_simulation_results-CTTAvg-251007-uncoupled_no_baiting-322_end.pkl', 'rb') as fi:
#     dict_simulation_results_ctt_avg_uncoupled_no_baiting_322_end = pickle.load(fi)


# # merge all CTTAvg simulation results
# dict_simulation_results_ctt_avg_all = {
#     **dict_simulation_results_ctt_avg_coupled_baiting,
#     **dict_simulation_results_ctt_avg_uncoupled_baiting,
#     **dict_simulation_results_ctt_avg_uncoupled_no_baiting_322_end
# }

# dict_simulation_results_ctt_avg_all.keys()

In [ ]:
# load simulation results: ctt_mean_reset
with open("/root/capsule/analysis_result/dict_simulation_results-CTTMeanReset-251014.pkl", 'rb') as f:
    dict_simulation_results_ctt_mean_reset = pickle.load(f)

In [ ]:
# merge into dict_simulation_results
dict_simulation_results = {
    **dict_simulation_results,
    **dict_simulation_results_ctt_avg,
    # **dict_simulation_results_ctt_mean_reset
}

# check keys
dict_simulation_results.keys()

## history dependency analysis

In [ ]:
def compute_switch_probabilities_by_history_pattern(df_session_history, max_trials_back=3):
    """
    Compute switch probabilities based on history patterns of choices and rewards.
    
    Parameters:
    -----------
    df_session_history : pd.DataFrame
        DataFrame where each row is a session with 'choice_history' and 'reward_history' columns
    max_trials_back : int
        Maximum number of trials to look back for pattern analysis
        
    Returns:
    --------
    results : dict
        Nested dictionary containing switch probabilities at different aggregation levels
    """
    
    def encode_trial(choice, reward):
        """Encode a single trial as L/R for rewarded, l/r for unrewarded."""
        if choice == 0:  # left choice
            return 'L' if reward > 0 else 'l'
        else:  # right choice
            return 'R' if reward > 0 else 'r'
    
    def pattern_to_abstract(pattern):
        """
        Convert specific pattern (L,R,l,r) to abstract pattern (A,B,a,b) 
        where the first choice side encountered becomes A/a, the other becomes B/b.
        Uppercase for rewarded, lowercase for unrewarded.
        """
        if not pattern:
            return ''
        
        # Find the first choice side (ignoring reward)
        first_side = pattern[0].upper()
        
        # Create mapping: first side -> A, other side -> B
        if first_side == 'L':
            mapping = {'L': 'A', 'l': 'a', 'R': 'B', 'r': 'b'}
        else:  # first_side == 'R'
            mapping = {'L': 'B', 'l': 'b', 'R': 'A', 'r': 'a'}
        
        return ''.join([mapping[char] for char in pattern])
    
    # Initialize results structure
    results = {
        'detailed': {},  # Full pattern breakdown (L, R, l, r)
        'abstract': {},  # Side-abstracted patterns (A, B, a, b)
        'trial_data': []  # Individual trial records for further analysis
    }
    
    for n_back in range(1, max_trials_back + 1):
        results['detailed'][n_back] = {}
        results['abstract'][n_back] = {}
    
    # Process each session
    for session_idx, row in df_session_history.iterrows():
        choices = row['choice_history']
        rewards = row['reward_history']
        
        if choices is None or rewards is None:
            continue
            
        choices = np.asarray(choices)
        rewards = np.asarray(rewards)
        
        # Remove NaN values
        valid_mask = ~(np.isnan(choices) | np.isnan(rewards))
        choices = choices[valid_mask]
        rewards = rewards[valid_mask]
        
        if len(choices) < max_trials_back + 1:
            continue
        
        # Encode all trials
        encoded_trials = [encode_trial(c, r) for c, r in zip(choices, rewards)]
        
        # Analyze patterns for different lookback windows
        for n_back in range(1, max_trials_back + 1):
            # Start from trial n_back (so we have history to look at)
            for trial_idx in range(n_back, len(encoded_trials)):
                # Get history pattern
                history_pattern = ''.join(encoded_trials[trial_idx - n_back:trial_idx])
                current_choice = choices[trial_idx]
                prev_choice = choices[trial_idx - 1]
                
                # Determine if this is a switch
                is_switch = current_choice != prev_choice
                
                # Create trial record for detailed analysis
                trial_record = {
                    'session_idx': session_idx,
                    'subject_id': row.get('subject_id', None),
                    'session_date': row.get('session_date', None),
                    'curriculum_name': row.get('curriculum_name', None),
                    'trial_idx': trial_idx,
                    'n_back': n_back,
                    'history_pattern': history_pattern,
                    'abstract_pattern': pattern_to_abstract(history_pattern),
                    'is_switch': is_switch,
                    'current_choice': current_choice,
                    'prev_choice': prev_choice
                }
                results['trial_data'].append(trial_record)
                
                # Update pattern counts for detailed patterns
                if history_pattern not in results['detailed'][n_back]:
                    results['detailed'][n_back][history_pattern] = {
                        'switches': 0, 'stays': 0, 'total': 0
                    }
                
                results['detailed'][n_back][history_pattern]['total'] += 1
                if is_switch:
                    results['detailed'][n_back][history_pattern]['switches'] += 1
                else:
                    results['detailed'][n_back][history_pattern]['stays'] += 1
                
                # Update pattern counts for abstract patterns
                abstract_pattern = pattern_to_abstract(history_pattern)
                if abstract_pattern not in results['abstract'][n_back]:
                    results['abstract'][n_back][abstract_pattern] = {
                        'switches': 0, 'stays': 0, 'total': 0
                    }
                
                results['abstract'][n_back][abstract_pattern]['total'] += 1
                if is_switch:
                    results['abstract'][n_back][abstract_pattern]['switches'] += 1
                else:
                    results['abstract'][n_back][abstract_pattern]['stays'] += 1
    
    # Calculate switch probabilities
    for n_back in range(1, max_trials_back + 1):
        # Detailed patterns
        for pattern in results['detailed'][n_back]:
            counts = results['detailed'][n_back][pattern]
            counts['switch_probability'] = counts['switches'] / counts['total'] if counts['total'] > 0 else np.nan
            counts['switch_probability_sem'] = np.sqrt(
                counts['switch_probability'] * (1 - counts['switch_probability']) / counts['total']
            ) if counts['total'] > 0 and not np.isnan(counts['switch_probability']) else np.nan
        
        # Abstract patterns
        for pattern in results['abstract'][n_back]:
            counts = results['abstract'][n_back][pattern]
            counts['switch_probability'] = counts['switches'] / counts['total'] if counts['total'] > 0 else np.nan
            counts['switch_probability_sem'] = np.sqrt(
                counts['switch_probability'] * (1 - counts['switch_probability']) / counts['total']
            ) if counts['total'] > 0 and not np.isnan(counts['switch_probability']) else np.nan
    
    return results


def analyze_switch_patterns_by_level(pattern_results, analysis_level='all'):
    """
    Analyze switch pattern results at different aggregation levels.
    
    Parameters:
    -----------
    pattern_results : dict
        Output from compute_switch_probabilities_by_history_pattern
    analysis_level : str
        'all', 'subject', or 'session' level analysis
        
    Returns:
    --------
    analysis_results : dict
        Aggregated results at specified level
    """
    trial_data = pd.DataFrame(pattern_results['trial_data'])
    
    if analysis_level == 'all':
        # Already computed in the main function
        return pattern_results
    
    elif analysis_level == 'subject':
        # Group by subject and recompute
        subject_results = {}
        
        for subject_id, subject_trials in trial_data.groupby('subject_id'):
            subject_results[subject_id] = {}
            
            for n_back in range(1, 4):
                subject_results[subject_id][n_back] = {
                    'detailed': {}, 'abstract': {}
                }
                
                n_back_trials = subject_trials[subject_trials['n_back'] == n_back]
                
                for pattern_type in ['history_pattern', 'abstract_pattern']:
                    pattern_dict = subject_results[subject_id][n_back][
                        'detailed' if pattern_type == 'history_pattern' else 'abstract'
                    ]
                    
                    for pattern, pattern_trials in n_back_trials.groupby(pattern_type):
                        switches = pattern_trials['is_switch'].sum()
                        total = len(pattern_trials)
                        stays = total - switches
                        
                        pattern_dict[pattern] = {
                            'switches': switches,
                            'stays': stays,
                            'total': total,
                            'switch_probability': switches / total if total > 0 else np.nan,
                            'switch_probability_sem': np.sqrt(
                                (switches / total) * (1 - switches / total) / total
                            ) if total > 0 else np.nan
                        }
        
        return subject_results
    
    elif analysis_level == 'session':
        # Group by session and recompute
        session_results = {}
        
        for (subject_id, session_date), session_trials in trial_data.groupby(['subject_id', 'session_date']):
            session_key = (subject_id, session_date)
            session_results[session_key] = {}
            
            for n_back in range(1, 4):
                session_results[session_key][n_back] = {
                    'detailed': {}, 'abstract': {}
                }
                
                n_back_trials = session_trials[session_trials['n_back'] == n_back]
                
                for pattern_type in ['history_pattern', 'abstract_pattern']:
                    pattern_dict = session_results[session_key][n_back][
                        'detailed' if pattern_type == 'history_pattern' else 'abstract'
                    ]
                    
                    for pattern, pattern_trials in n_back_trials.groupby(pattern_type):
                        switches = pattern_trials['is_switch'].sum()
                        total = len(pattern_trials)
                        stays = total - switches
                        
                        pattern_dict[pattern] = {
                            'switches': switches,
                            'stays': stays,
                            'total': total,
                            'switch_probability': switches / total if total > 0 else np.nan,
                            'switch_probability_sem': np.sqrt(
                                (switches / total) * (1 - switches / total) / total
                            ) if total > 0 else np.nan
                        }
        
        return session_results


# def plot_switch_probabilities_by_pattern(pattern_results, n_back=1, pattern_type='detailed'):
#     """
#     Plot switch probabilities for different history patterns.
    
#     Parameters:
#     -----------
#     pattern_results : dict
#         Output from compute_switch_probabilities_by_history_pattern
#     n_back : int
#         Number of trials back to analyze
#     pattern_type : str
#         'detailed' or 'abstract'
#     """
#     plt.figure(figsize=(12, 8), dpi=300)
    
#     # Get the relevant pattern data
#     pattern_data = pattern_results[pattern_type][n_back]
    
#     # Sort patterns for consistent display
#     patterns = sorted(pattern_data.keys())
    
#     # Extract data for plotting
#     probs = []
#     sems = []
#     totals = []
    
#     for pattern in patterns:
#         probs.append(pattern_data[pattern]['switch_probability'])
#         sems.append(pattern_data[pattern]['switch_probability_sem'])
#         totals.append(pattern_data[pattern]['total'])
    
#     # Create bar plot
#     x_pos = np.arange(len(patterns))
#     bars = plt.bar(x_pos, probs, yerr=sems, capsize=5, alpha=0.7)
    
#     # Add sample size annotations
#     for i, (bar, n) in enumerate(zip(bars, totals)):
#         height = bar.get_height()
#         if not np.isnan(height):
#             plt.text(bar.get_x() + bar.get_width()/2., height + sems[i] + 0.01,
#                     f'n={n}', ha='center', va='bottom', fontsize=10, rotation=45)
    
#     plt.xlabel('History Pattern')
#     plt.ylabel('Switch Probability')
#     plt.title(f'Switch Probability by {pattern_type.capitalize()} History Pattern ({n_back} trial{"s" if n_back > 1 else ""} back)')
#     plt.xticks(x_pos, patterns, rotation=45)
#     plt.grid(True, alpha=0.3)
#     plt.tight_layout()
#     plt.show()
    
#     # Print numerical results
#     print(f"\nSwitch probabilities for {pattern_type} patterns ({n_back} trial{'s' if n_back > 1 else ''} back):")
#     print("-" * 60)
#     for pattern in patterns:
#         data = pattern_data[pattern]
#         if not np.isnan(data['switch_probability']):
#             print(f"{pattern:>8}: {data['switch_probability']:.3f} ± {data['switch_probability_sem']:.3f} (n={data['total']})")

In [ ]:
def plot_switch_probabilities_by_pattern(pattern_results, n_back=1, pattern_type='detailed'):
    """
    Plot switch probabilities for different history patterns.
    
    Parameters:
    -----------
    pattern_results : dict
        Output from compute_switch_probabilities_by_history_pattern
    n_back : int
        Number of trials back to analyze
    pattern_type : str
        'detailed' or 'abstract'
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 12), dpi=300)
    
    # Get the relevant pattern data
    pattern_data = pattern_results[pattern_type][n_back]
    
    # Extract data with valid probabilities
    pattern_info = []
    for pattern in pattern_data.keys():
        prob = pattern_data[pattern]['switch_probability']
        sem = pattern_data[pattern]['switch_probability_sem']
        total = pattern_data[pattern]['total']
        if not np.isnan(prob):  # Only include patterns with valid probabilities
            pattern_info.append((pattern, prob, sem, total))
    
    # Plot 1: Unsorted (alphabetical)
    pattern_info_unsorted = sorted(pattern_info, key=lambda x: x[0])  # Sort by pattern name
    patterns_unsorted = [info[0] for info in pattern_info_unsorted]
    probs_unsorted = [info[1] for info in pattern_info_unsorted]
    sems_unsorted = [info[2] for info in pattern_info_unsorted]
    totals_unsorted = [info[3] for info in pattern_info_unsorted]
    
    x_pos = np.arange(len(patterns_unsorted))
    bars1 = ax1.bar(x_pos, probs_unsorted, yerr=sems_unsorted, capsize=5, alpha=0.7)
    
    # Add sample size annotations for plot 1
    for i, (bar, n) in enumerate(zip(bars1, totals_unsorted)):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + sems_unsorted[i] + 0.01,
                f'n={n}', ha='center', va='bottom', fontsize=10, rotation=45)
    
    ax1.set_xlabel('History Pattern', fontsize=14)
    ax1.set_ylabel('Switch Probability', fontsize=14)
    ax1.set_title(f'Unsorted - Switch Probability by {pattern_type.capitalize()} History Pattern\n({n_back} trial{"s" if n_back > 1 else ""} back)', fontsize=16)
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(patterns_unsorted, rotation=45, fontsize=12)
    ax1.tick_params(axis='y', labelsize=12)
    ax1.set_ylim(0, 0.7)
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Sorted by switch probability (low to high)
    pattern_info_sorted = sorted(pattern_info, key=lambda x: x[1])  # Sort by probability
    patterns_sorted = [info[0] for info in pattern_info_sorted]
    probs_sorted = [info[1] for info in pattern_info_sorted]
    sems_sorted = [info[2] for info in pattern_info_sorted]
    totals_sorted = [info[3] for info in pattern_info_sorted]
    
    x_pos = np.arange(len(patterns_sorted))
    bars2 = ax2.bar(x_pos, probs_sorted, yerr=sems_sorted, capsize=5, alpha=0.7)
    
    # Add sample size annotations for plot 2
    for i, (bar, n) in enumerate(zip(bars2, totals_sorted)):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + sems_sorted[i] + 0.01,
                f'n={n}', ha='center', va='bottom', fontsize=10, rotation=45)
    
    ax2.set_xlabel('History Pattern', fontsize=14)
    ax2.set_ylabel('Switch Probability', fontsize=14)
    ax2.set_title(f'Sorted - Switch Probability by {pattern_type.capitalize()} History Pattern\n({n_back} trial{"s" if n_back > 1 else ""} back)', fontsize=16)
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(patterns_sorted, rotation=45, fontsize=12)
    ax2.tick_params(axis='y', labelsize=12)
    ax2.set_ylim(0, 0.7)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print numerical results (sorted order)
    print(f"\nSwitch probabilities for {pattern_type} patterns ({n_back} trial{'s' if n_back > 1 else ''} back):")
    print("-" * 60)
    for pattern, prob, sem, total in pattern_info_sorted:
        print(f"{pattern:>8}: {prob:.3f} ± {sem:.3f} (n={total})")

### analyze mice data

In [ ]:
# choose task type to analyze

# task_type = 'Uncoupled Baiting'
# task_type = 'Uncoupled Without Baiting'
task_type = 'Coupled Baiting'

In [ ]:
# Compute switch probabilities by history pattern

# Filter sessions for the current task type and later stages
df_session_history_task_type = df_session_history[
    (df_session_history['curriculum_name'] == task_type) &
    (df_session_history['current_stage_actual'].isin(later_stages))]


pattern_results = compute_switch_probabilities_by_history_pattern(
    df_session_history_task_type, 
    max_trials_back=3
)

In [ ]:
# Plot results for different pattern types and lookback windows
for n_back in [1, 2, 3]:
    for pattern_type in ['detailed', 'abstract']:
        plot_switch_probabilities_by_pattern(pattern_results, n_back=n_back, pattern_type=pattern_type)

In [ ]:
# Analyze at subject level
subject_level_results = analyze_switch_patterns_by_level(pattern_results, analysis_level='subject')

# Analyze at session level  
# session_results = analyze_switch_patterns_by_level(pattern_results, analysis_level='session')

# Access individual trial data for custom analyses
# trial_df = pd.DataFrame(pattern_results['trial_data'])
# print(f"Total trials analyzed: {len(trial_df)}")
# print(trial_df.head())

### subject level variability

In [ ]:
def extract_per_animal_switch_probabilities(subject_results, pattern_type='abstract', min_trials=5):
    """
    Extract per-animal switch probabilities for each history pattern.
    
    Parameters:
    -----------
    subject_results : dict
        Output from analyze_switch_patterns_by_level with analysis_level='subject'
    pattern_type : str
        'detailed' or 'abstract' patterns
    min_trials : int
        Minimum number of trials required to include an animal for a pattern
        
    Returns:
    --------
    pattern_animal_data : dict
        Nested dictionary: {n_back: {pattern: {'subject_ids': [], 'switch_probabilities': [], 'n_trials': []}}}
    """
    pattern_animal_data = {}
    
    for n_back in [1, 2, 3]:
        pattern_animal_data[n_back] = {}
        
        for subject_id, subject_data in subject_results.items():
            if n_back not in subject_data:
                continue
                
            patterns = subject_data[n_back][pattern_type]
            
            for pattern, data in patterns.items():
                if data['total'] >= min_trials and not np.isnan(data['switch_probability']):
                    if pattern not in pattern_animal_data[n_back]:
                        pattern_animal_data[n_back][pattern] = {
                            'subject_ids': [],
                            'switch_probabilities': [],
                            'n_trials': []
                        }
                    
                    pattern_animal_data[n_back][pattern]['subject_ids'].append(subject_id)
                    pattern_animal_data[n_back][pattern]['switch_probabilities'].append(data['switch_probability'])
                    pattern_animal_data[n_back][pattern]['n_trials'].append(data['total'])
    
    return pattern_animal_data


def plot_per_animal_switch_probability_histograms(subject_results, task_type, 
                                                  pattern_type='abstract', 
                                                  min_trials=5, bins=15):
    """
    Plot histograms of per-animal switch probabilities for each history pattern.
    Creates one subplot per pattern, organized by n_back (1, 2, 3 trials back).
    
    Parameters:
    -----------
    subject_results : dict
        Output from analyze_switch_patterns_by_level with analysis_level='subject'
    task_type : str
        Task type being analyzed
    pattern_type : str
        'detailed' or 'abstract' patterns
    min_trials : int
        Minimum number of trials required to include an animal for a pattern
    bins : int or str
        Number of bins for histogram
        
    Returns:
    --------
    pattern_animal_data : dict
        Dictionary containing per-animal data for each pattern
    df_animal_data : pd.DataFrame
        DataFrame containing all per-animal data
    """
    # Extract per-animal data
    pattern_animal_data = extract_per_animal_switch_probabilities(
        subject_results, pattern_type=pattern_type, min_trials=min_trials
    )
    
    # Count total patterns for each n_back
    patterns_by_nback = {n_back: sorted(pattern_animal_data[n_back].keys()) 
                         for n_back in [1, 2, 3]}
    
    total_patterns = sum(len(patterns) for patterns in patterns_by_nback.values())
    
    if total_patterns == 0:
        print("No patterns with sufficient data found.")
        return None, None
    
    print(f"Found {total_patterns} patterns with sufficient data:")
    for n_back in [1, 2, 3]:
        print(f"  {n_back} trial{'s' if n_back > 1 else ''} back: {len(patterns_by_nback[n_back])} patterns")
    
    # Create separate figure for each n_back
    for n_back in [1, 2, 3]:
        patterns = patterns_by_nback[n_back]
        n_patterns = len(patterns)
        
        if n_patterns == 0:
            continue
        
        # Determine grid layout
        n_cols = min(4, n_patterns)
        n_rows = int(np.ceil(n_patterns / n_cols))
        
        # Create figure
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows), dpi=300)
        
        # Flatten axes array for easier indexing
        if n_patterns == 1:
            axes = np.array([axes])
        else:
            axes = axes.flatten()
        
        # Plot histogram for each pattern
        for idx, pattern in enumerate(patterns):
            ax = axes[idx]
            
            probs = pattern_animal_data[n_back][pattern]['switch_probabilities']
            n_animals = len(probs)
            
            # Create histogram
            n_hist, bins_edges, patches = ax.hist(probs, bins=bins, alpha=0.7, 
                                              color='steelblue', edgecolor='black')
            
            # Add mean and std lines
            mean_prob = np.mean(probs)
            std_prob = np.std(probs)
            
            ax.axvline(mean_prob, color='red', linestyle='--', linewidth=2, 
                       label=f'Mean: {mean_prob:.3f}')
            ax.axvline(mean_prob - std_prob, color='orange', linestyle=':', linewidth=1.5,
                       label=f'±1 SD: {std_prob:.3f}')
            ax.axvline(mean_prob + std_prob, color='orange', linestyle=':', linewidth=1.5)
            
            # Set labels and title
            ax.set_xlabel('Switch Probability', fontsize=11)
            ax.set_ylabel('Number of Animals', fontsize=11)
            ax.set_title(f'Pattern: {pattern}\n(n={n_animals} animals)', 
                        fontsize=12, fontweight='bold')
            ax.legend(fontsize=9, loc='best')
            ax.grid(True, alpha=0.3, axis='y')
            ax.set_xlim(0, 1)
            
            # Set y-axis to integer ticks
            ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
        
        # Hide extra subplots
        for idx in range(n_patterns, len(axes)):
            axes[idx].set_visible(False)
        
        # Add main title
        plt.suptitle(f'Per-Animal Switch Probability Distributions: {task_type}\n'
                     f'{pattern_type.capitalize()} Patterns ({n_back} trial{"s" if n_back > 1 else ""} back)',
                     fontsize=16, fontweight='bold', y=1.00)
        
        plt.tight_layout()
        plt.show()
        
        # Print summary statistics for this n_back
        print(f"\nSummary Statistics for {n_back} trial{'s' if n_back > 1 else ''} back:")
        print("="*80)
        for pattern in patterns:
            probs = pattern_animal_data[n_back][pattern]['switch_probabilities']
            print(f"\nPattern {pattern}:")
            print(f"  N animals: {len(probs)}")
            print(f"  Mean: {np.mean(probs):.3f}")
            print(f"  Std: {np.std(probs):.3f}")
            print(f"  CV: {np.std(probs)/np.mean(probs):.3f}")
            print(f"  Median: {np.median(probs):.3f}")
            print(f"  Min: {np.min(probs):.3f}")
            print(f"  Max: {np.max(probs):.3f}")
            print(f"  Q1 (25%): {np.percentile(probs, 25):.3f}")
            print(f"  Q3 (75%): {np.percentile(probs, 75):.3f}")
            print(f"  IQR: {np.percentile(probs, 75) - np.percentile(probs, 25):.3f}")
    
    # Create DataFrame with all animal data
    animal_data_list = []
    for n_back in [1, 2, 3]:
        for pattern in patterns_by_nback[n_back]:
            for subject_id, prob, n_trials in zip(
                pattern_animal_data[n_back][pattern]['subject_ids'],
                pattern_animal_data[n_back][pattern]['switch_probabilities'],
                pattern_animal_data[n_back][pattern]['n_trials']
            ):
                animal_data_list.append({
                    'subject_id': subject_id,
                    'n_back': n_back,
                    'pattern': pattern,
                    'switch_probability': prob,
                    'n_trials': n_trials
                })
    
    df_animal_data = pd.DataFrame(animal_data_list)
    
    return pattern_animal_data, df_animal_data


def analyze_per_animal_variability_all_tasks(df_session_history, task_types, later_stages,
                                            pattern_type='abstract', min_trials=5, bins=15):
    """
    Analyze per-animal switch probability variability for all task types.
    
    Parameters:
    -----------
    df_session_history : pd.DataFrame
        Real mice session data
    task_types : list
        List of task types to analyze
    later_stages : list
        List of later stage names to filter for
    pattern_type : str
        'detailed' or 'abstract' patterns
    min_trials : int
        Minimum number of trials required to include an animal for a pattern
    bins : int or str
        Number of bins for histogram
        
    Returns:
    --------
    all_animal_data : dict
        Dictionary containing per-animal data for all task types
    """
    all_animal_data = {}
    
    for task_type in task_types:
        print(f"\n{'='*80}")
        print(f"Analyzing per-animal variability for {task_type}")
        print(f"{'='*80}")
        
        # Filter data for this task type
        df_mice_task = df_session_history[
            (df_session_history['curriculum_name'] == task_type) &
            (df_session_history['current_stage_actual'].isin(later_stages))
        ]
        
        # Compute population-level results
        pattern_results = compute_switch_probabilities_by_history_pattern(
            df_mice_task, max_trials_back=3
        )
        
        # Get subject-level breakdown
        subject_results = analyze_switch_patterns_by_level(
            pattern_results, analysis_level='subject'
        )
        
        # Plot histograms and get data
        pattern_animal_data, df_animal_data = plot_per_animal_switch_probability_histograms(
            subject_results, task_type, 
            pattern_type=pattern_type, 
            min_trials=min_trials, 
            bins=bins
        )
        
        all_animal_data[task_type] = {
            'pattern_animal_data': pattern_animal_data,
            'df_animal_data': df_animal_data
        }
    
    return all_animal_data

In [ ]:
# Analyze per-animal variability for all task types
task_types = [
    'Uncoupled Without Baiting', 
    'Uncoupled Baiting',
    'Coupled Baiting'
]

all_animal_variability = analyze_per_animal_variability_all_tasks(
    df_session_history,
    task_types,
    later_stages,
    pattern_type='abstract',
    min_trials=5,
    bins=15
)

In [ ]:
# Using the subject_results from your notebook
# First, make sure you have subject_results computed:
# subject_results = analyze_switch_patterns_by_level(pattern_results, analysis_level='subject')

# Plot histograms for a specific task
task_type = 'Coupled Baiting'
pattern_animal_data, df_animal_data = plot_per_animal_switch_probability_histograms(
    subject_results, 
    task_type=task_type,
    pattern_type='abstract',
    min_trials=5,
    bins=15
)

# View the DataFrame with per-animal data
print(df_animal_data.head())

In [ ]:
# Analyze for a specific task
task_type = 'Coupled Baiting'

# First get the data and compute subject-level results
df_mice_task = df_session_history[
    (df_session_history['curriculum_name'] == task_type) &
    (df_session_history['current_stage_actual'].isin(later_stages))
]

pattern_results = compute_switch_probabilities_by_history_pattern(
    df_mice_task, max_trials_back=3
)

subject_results = analyze_switch_patterns_by_level(
    pattern_results, analysis_level='subject'
)

# Then plot histograms
pattern_animal_data, df_animal_data = plot_per_animal_switch_probability_histograms(
    subject_results, 
    task_type=task_type,
    pattern_type='abstract',
    min_trials=5,
    bins=15
)

### analyze simulation data

In [ ]:
# choose task type to analyze

# task_type = 'Uncoupled Baiting'
# task_type = 'Uncoupled Without Baiting'
task_type = 'Coupled Baiting'

# model_type = 'ForagingCompareThreshold'
# model_type = 'QLearning_L2F1_softmax'
model_type = 'QLearning_L1F1_CK1_softmax'


df_sim_model_task = dict_simulation_results[(model_type, task_type)]

In [ ]:
# Compute switch probabilities by history pattern
pattern_results = compute_switch_probabilities_by_history_pattern(
    df_sim_model_task, 
    max_trials_back=3
)

In [ ]:
# Plot results for different pattern types and lookback windows
for n_back in [1, 2, 3]:
    for pattern_type in ['detailed', 'abstract']:
        plot_switch_probabilities_by_pattern(pattern_results, n_back=n_back, pattern_type=pattern_type)

## comparison plot

In [ ]:
def calculate_all_switch_probabilities(df_session_history, dict_simulation_results, task_type, model_types, later_stages):
    """
    Calculate switch probabilities for both mice and simulated data for all model types.
    
    Parameters:
    -----------
    df_session_history : pd.DataFrame
        Real mice session data
    dict_simulation_results : dict
        Dictionary of simulation results
    task_type : str
        Task type to analyze
    model_types : list
        List of model types to analyze
    later_stages : list
        List of later stage names to filter for
        
    Returns:
    --------
    results : dict
        Dictionary containing switch probability results for mice and all models
    """
    results = {}
    
    # Calculate mice data results
    print(f"Calculating switch probabilities for mice data - {task_type}")
    df_mice_task = df_session_history[
        (df_session_history['curriculum_name'] == task_type) &
        (df_session_history['current_stage_actual'].isin(later_stages))
    ]
    
    results['mice'] = compute_switch_probabilities_by_history_pattern(
        df_mice_task, max_trials_back=3
    )
    
    # Calculate simulation results for each model
    for model_type in model_types:
        print(f"Calculating switch probabilities for {model_type} - {task_type}")
        df_sim_model_task = dict_simulation_results[(model_type, task_type)]
        
        results[model_type] = compute_switch_probabilities_by_history_pattern(
            df_sim_model_task, max_trials_back=3
        )
    
    return results


def plot_mice_vs_model_comparison(all_results, task_type, model_types, pattern_type='abstract', min_trials=10):
    """
    Create comparison plots between mice and model data for history-dependent switch probabilities.
    
    Parameters:
    -----------
    all_results : dict
        Results from calculate_all_switch_probabilities
    task_type : str
        Task type being analyzed
    model_types : list
        List of model types to compare
    pattern_type : str
        'detailed' or 'abstract' patterns
    min_trials : int
        Minimum number of trials required to include a pattern in the plot
    """
    fig, axes = plt.subplots(len(model_types), 3, figsize=(20, 6*len(model_types)), dpi=300)
    
    # Ensure axes is 2D even for single model
    if len(model_types) == 1:
        axes = axes.reshape(1, -1)
    
    # Create a comprehensive color palette for up to 32 patterns (for 3 trials back)
    colors = plt.cm.tab10(np.linspace(0, 1, 10))  # First 10 colors from tab10
    colors = np.vstack([
        colors,
        plt.cm.Set3(np.linspace(0, 1, 12)),  # Next 12 colors from Set3
        plt.cm.Pastel1(np.linspace(0, 1, 9)),  # Next 9 colors from Pastel1
        plt.cm.Dark2(np.linspace(0, 1, 8))  # Last 8 colors from Dark2
    ])  # Total: 39 colors
    
    # Collect all unique patterns across all models and n_back values for consistent coloring
    all_patterns = set()
    for model_type in model_types:
        for n_back in [1, 2, 3]:
            mice_patterns = all_results['mice'][pattern_type][n_back]
            model_patterns = all_results[model_type][pattern_type][n_back]
            
            for pattern in mice_patterns.keys():
                if (pattern in model_patterns and 
                    mice_patterns[pattern]['total'] >= min_trials and 
                    model_patterns[pattern]['total'] >= min_trials and
                    not np.isnan(mice_patterns[pattern]['switch_probability']) and
                    not np.isnan(model_patterns[pattern]['switch_probability'])):
                    all_patterns.add(pattern)
    
    # Sort patterns for consistent ordering
    all_patterns = sorted(list(all_patterns))
    pattern_colors = {pattern: colors[i % len(colors)] for i, pattern in enumerate(all_patterns)}
    
    # Track legend handles and labels for the bottom legend
    legend_handles = {}
    legend_labels = {}
    
    for model_idx, model_type in enumerate(model_types):
        mice_results = all_results['mice']
        model_results = all_results[model_type]
        
        for n_back in [1, 2, 3]:
            ax = axes[model_idx, n_back-1]
            
            # Get patterns that exist in both mice and model data with sufficient trials
            mice_patterns = mice_results[pattern_type][n_back]
            model_patterns = model_results[pattern_type][n_back]
            
            # Find common patterns with sufficient data
            common_patterns = []
            for pattern in mice_patterns.keys():
                if (pattern in model_patterns and 
                    mice_patterns[pattern]['total'] >= min_trials and 
                    model_patterns[pattern]['total'] >= min_trials and
                    not np.isnan(mice_patterns[pattern]['switch_probability']) and
                    not np.isnan(model_patterns[pattern]['switch_probability'])):
                    common_patterns.append(pattern)
            
            if not common_patterns:
                ax.text(0.5, 0.5, 'No common patterns\nwith sufficient data', 
                       transform=ax.transAxes, ha='center', va='center', fontsize=12)
                ax.set_title(f'{model_type}\n{n_back} trial{"s" if n_back > 1 else ""} back')
                continue
            
            # Extract data for plotting
            mice_probs = []
            model_probs = []
            mice_sems = []
            model_sems = []
            
            for pattern in common_patterns:
                mice_prob = mice_patterns[pattern]['switch_probability']
                model_prob = model_patterns[pattern]['switch_probability']
                mice_sem = mice_patterns[pattern]['switch_probability_sem']
                model_sem = model_patterns[pattern]['switch_probability_sem']
                
                mice_probs.append(mice_prob)
                model_probs.append(model_prob)
                mice_sems.append(mice_sem)
                model_sems.append(model_sem)
                
                # Plot point with error bars
                handle = ax.errorbar(mice_prob, model_prob, 
                           xerr=mice_sem, yerr=model_sem,
                           marker='o', markersize=8, 
                           color=pattern_colors[pattern],
                           capsize=3, alpha=0.8, linewidth=2)
                
                # Store legend information (only need one handle per pattern)
                if pattern not in legend_handles:
                    legend_handles[pattern] = handle
                    legend_labels[pattern] = pattern
            
            # Add unity line
            if mice_probs and model_probs:
                max_val = max(max(mice_probs), max(model_probs))
                min_val = min(min(mice_probs), min(model_probs))
                
                ax.plot([min_val, max_val], [min_val, max_val], 
                       'k--', alpha=0.5, linewidth=2)
                
                # Set equal aspect ratio and limits
                ax.set_aspect('equal')
                buffer = 0.05
                ax.set_xlim(min_val - buffer, max_val + buffer)
                ax.set_ylim(min_val - buffer, max_val + buffer)
            
            # Set labels and title
            ax.set_xlabel('Mice Switch Probability', fontsize=12)
            ax.set_ylabel('Model Switch Probability', fontsize=12)
            ax.set_title(f'{model_type}\n{n_back} trial{"s" if n_back > 1 else ""} back', fontsize=14)
            ax.grid(True, alpha=0.3)
    
    # Add main title
    plt.suptitle(f'Mice vs Model Comparison: {task_type}\n{pattern_type.capitalize()} Patterns', 
                 fontsize=16, y=1.0)
    
    # Create legend at the bottom with multiple columns
    # Sort patterns for legend
    sorted_patterns = sorted(legend_handles.keys())
    handles = [legend_handles[pattern] for pattern in sorted_patterns]
    labels = [legend_labels[pattern] for pattern in sorted_patterns]
    
    # Determine number of columns for legend based on number of patterns
    n_patterns = len(sorted_patterns)
    if n_patterns <= 8:
        ncol = 4
    elif n_patterns <= 16:
        ncol = 8
    else:
        ncol = 16  # For up to 32 patterns, use 16 columns (2 rows)
    
    # Add legend below the plots
    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.0), 
              ncol=ncol, fontsize=10, frameon=True, fancybox=True, shadow=True)
    
    # Adjust layout to make room for legend
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15 if n_patterns > 16 else 0.1)
    plt.show()
    
    # Print correlation statistics
    print(f"\nCorrelation statistics for {task_type} - {pattern_type} patterns:")
    print("="*80)
    
    for model_idx, model_type in enumerate(model_types):
        mice_results = all_results['mice']
        model_results = all_results[model_type]
        
        print(f"\n{model_type}:")
        print("-"*40)
        
        for n_back in [1, 2, 3]:
            mice_patterns = mice_results[pattern_type][n_back]
            model_patterns = model_results[pattern_type][n_back]
            
            # Find common patterns
            common_patterns = []
            mice_probs = []
            model_probs = []
            
            for pattern in mice_patterns.keys():
                if (pattern in model_patterns and 
                    mice_patterns[pattern]['total'] >= min_trials and 
                    model_patterns[pattern]['total'] >= min_trials and
                    not np.isnan(mice_patterns[pattern]['switch_probability']) and
                    not np.isnan(model_patterns[pattern]['switch_probability'])):
                    
                    common_patterns.append(pattern)
                    mice_probs.append(mice_patterns[pattern]['switch_probability'])
                    model_probs.append(model_patterns[pattern]['switch_probability'])
            
            if len(mice_probs) > 1:
                correlation = np.corrcoef(mice_probs, model_probs)[0, 1]
                rmse = np.sqrt(np.mean((np.array(mice_probs) - np.array(model_probs))**2))
                print(f"  {n_back} trials back: r={correlation:.3f}, RMSE={rmse:.3f}, n_patterns={len(common_patterns)}")
            else:
                print(f"  {n_back} trials back: insufficient data")


def generate_task_comparison_figures(df_session_history, dict_simulation_results, 
                                   task_type, model_types, later_stages, 
                                   pattern_type='abstract', min_trials=10):
    """
    Generate comparison figures for a specific task type comparing mice vs all model types.
    
    Parameters:
    -----------
    df_session_history : pd.DataFrame
        Real mice session data
    dict_simulation_results : dict
        Dictionary of simulation results
    task_type : str
        Task type to analyze
    model_types : list
        List of model types to compare
    later_stages : list
        List of later stage names to filter for
    pattern_type : str
        'detailed' or 'abstract' patterns
    min_trials : int
        Minimum number of trials required to include a pattern
    """
    print(f"Generating comparison figures for task type: {task_type}")
    
    # Calculate switch probabilities for all data
    all_results = calculate_all_switch_probabilities(
        df_session_history, dict_simulation_results, 
        task_type, model_types, later_stages
    )
    
    # Create comparison plots
    plot_mice_vs_model_comparison(
        all_results, task_type, model_types, 
        pattern_type=pattern_type, min_trials=min_trials
    )
    
    return all_results

In [ ]:
# Example usage - generate figures for all task types

task_types = [
    # 'Uncoupled Baiting', 
    'Uncoupled Without Baiting', 
    'Coupled Baiting'
]
model_types = [
    'QLearning_L2F1_softmax', 'QLearning_L1F1_CK1_softmax', 'ForagingCompareThreshold', 
    'CTTAvg'
    # 'CTTMeanReset'
]

# Store all results for further analysis if needed
all_task_results = {}

for task_type in task_types:
    print(f"\n{'='*50}")
    print(f"Processing {task_type}")
    print(f"{'='*50}")
    
    all_task_results[task_type] = generate_task_comparison_figures(
        df_session_history, dict_simulation_results,
        task_type, model_types, later_stages,
        pattern_type='abstract',  # or 'detailed'
        min_trials=10
    )

In [ ]:
# Create comparison plots

for task_type in task_types:

    all_results = all_task_results[task_type]

    plot_mice_vs_model_comparison(
        all_results, task_type, model_types, 
        pattern_type='abstract', min_trials=10
    )

In [ ]:
# save all_task_results to a file for later analysis
import pickle
with open('../analysis_result/switch_probabilities_by_history_pattern.pkl', 'wb') as f:
    pickle.dump(all_task_results, f)

### subject level analysis

In [ ]:
def compute_subject_level_switch_probabilities(df_session_history, dict_simulation_results, 
                                               task_type, model_types, later_stages,
                                               pattern_type='abstract', min_trials=5):
    """
    Compute switch probabilities at subject level for both mice and simulated data.
    
    Parameters:
    -----------
    df_session_history : pd.DataFrame
        Real mice session data
    dict_simulation_results : dict
        Dictionary of simulation results
    task_type : str
        Task type to analyze
    model_types : list
        List of model types to analyze
    later_stages : list
        List of later stage names to filter for
    pattern_type : str
        'detailed' or 'abstract' patterns
    min_trials : int
        Minimum number of trials required to include a subject-pattern combination
        
    Returns:
    --------
    subject_results : dict
        Dictionary containing subject-level results for mice and all models
    """
    subject_results = {}
    
    # Get mice data for this task type
    df_mice_task = df_session_history[
        (df_session_history['curriculum_name'] == task_type) &
        (df_session_history['current_stage_actual'].isin(later_stages))
    ]
    
    # Get unique subjects
    unique_subjects = df_mice_task['subject_id'].unique()
    print(f"Found {len(unique_subjects)} unique subjects for {task_type}")
    
    # Compute switch probabilities for each subject (mice data)
    subject_results['mice'] = {}
    for subject_id in unique_subjects:
        df_subject = df_mice_task[df_mice_task['subject_id'] == subject_id]
        subject_results['mice'][subject_id] = compute_switch_probabilities_by_history_pattern(
            df_subject, max_trials_back=3
        )
    
    # Compute switch probabilities for each model
    for model_type in model_types:
        print(f"Processing {model_type} for {task_type}")
        df_sim_model_task = dict_simulation_results[(model_type, task_type)]
        
        subject_results[model_type] = {}
        for subject_id in unique_subjects:
            # Check if this subject has simulated data
            df_sim_subject = df_sim_model_task[df_sim_model_task['subject_id'] == subject_id]
            if len(df_sim_subject) > 0:
                subject_results[model_type][subject_id] = compute_switch_probabilities_by_history_pattern(
                    df_sim_subject, max_trials_back=3
                )
    
    return subject_results, unique_subjects


def plot_single_pattern_subject_level(subject_results, unique_subjects, task_type, 
                                      model_types, pattern, pattern_type='abstract', 
                                      min_trials=5):
    """
    Create a comparison figure for a single history pattern showing all subjects across models.
    
    Parameters:
    -----------
    subject_results : dict
        Results from compute_subject_level_switch_probabilities
    unique_subjects : array-like
        Array of unique subject IDs
    task_type : str
        Task type being analyzed
    model_types : list
        List of model types to compare
    pattern : str
        The specific history pattern to plot
    pattern_type : str
        'detailed' or 'abstract' patterns
    min_trials : int
        Minimum number of trials required to include a subject
    """
    # Determine n_back from pattern length
    n_back = len(pattern)
    
    # Create color map for subjects
    n_subjects = len(unique_subjects)
    subject_colors = plt.cm.tab20(np.linspace(0, 1, min(20, n_subjects)))
    if n_subjects > 20:
        subject_colors = np.vstack([
            subject_colors,
            plt.cm.tab20b(np.linspace(0, 1, min(20, n_subjects - 20)))
        ])
        if n_subjects > 40:
            subject_colors = np.vstack([
                subject_colors,
                plt.cm.tab20c(np.linspace(0, 1, n_subjects - 40))
            ])
    
    subject_color_map = {subject: subject_colors[i % len(subject_colors)] 
                        for i, subject in enumerate(unique_subjects)}
    
    # Create figure with subplots for each model
    n_models = len(model_types)
    n_cols = min(3, n_models)
    n_rows = int(np.ceil(n_models / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(7*n_cols, 6*n_rows), dpi=300)
    
    # Flatten axes array for easier indexing
    if n_models == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()
    
    # Hide extra subplots if any
    for idx in range(n_models, len(axes)):
        axes[idx].set_visible(False)
    
    # Track legend handles and labels
    legend_handles = []
    legend_labels = []
    
    for model_idx, model_type in enumerate(model_types):
        ax = axes[model_idx]
        
        # Collect data for all subjects
        mice_probs = []
        model_probs = []
        subjects_plotted = []
        
        for subject_id in unique_subjects:
            # Check if subject has data for both mice and model
            if (subject_id not in subject_results['mice'] or 
                subject_id not in subject_results[model_type]):
                continue
            
            mice_patterns = subject_results['mice'][subject_id][pattern_type][n_back]
            model_patterns = subject_results[model_type][subject_id][pattern_type][n_back]
            
            # Check if pattern exists with sufficient data
            if (pattern in mice_patterns and pattern in model_patterns and
                mice_patterns[pattern]['total'] >= min_trials and
                model_patterns[pattern]['total'] >= min_trials and
                not np.isnan(mice_patterns[pattern]['switch_probability']) and
                not np.isnan(model_patterns[pattern]['switch_probability'])):
                
                mice_prob = mice_patterns[pattern]['switch_probability']
                model_prob = model_patterns[pattern]['switch_probability']
                
                mice_probs.append(mice_prob)
                model_probs.append(model_prob)
                subjects_plotted.append(subject_id)
        
        if not mice_probs:
            ax.text(0.5, 0.5, f'Pattern {pattern}\nNo subjects with\nsufficient data', 
                   transform=ax.transAxes, ha='center', va='center', fontsize=12)
            ax.set_title(model_type, fontsize=14)
            continue
        
        # Plot each subject with their specific color
        for mice_prob, model_prob, subject_id in zip(mice_probs, model_probs, subjects_plotted):
            handle = ax.scatter(mice_prob, model_prob, 
                       s=80, alpha=0.7, 
                       color=subject_color_map[subject_id],
                       edgecolors='black', linewidths=1)
            
            # Store legend info (only once per subject)
            if model_idx == 0 and subject_id not in legend_labels:
                legend_handles.append(handle)
                legend_labels.append(subject_id)
        
        # Add unity line
        if mice_probs and model_probs:
            min_val = min(min(mice_probs), min(model_probs))
            max_val = max(max(mice_probs), max(model_probs))
            
            ax.plot([min_val, max_val], [min_val, max_val], 
                   'k--', alpha=0.5, linewidth=2)
            
            # Set equal aspect ratio and limits
            ax.set_aspect('equal')
            buffer = (max_val - min_val) * 0.1
            ax.set_xlim(min_val - buffer, max_val + buffer)
            ax.set_ylim(min_val - buffer, max_val + buffer)
        
        # Calculate and display correlation
        if len(mice_probs) > 1:
            correlation = np.corrcoef(mice_probs, model_probs)[0, 1]
            rmse = np.sqrt(np.mean((np.array(mice_probs) - np.array(model_probs))**2))
            ax.text(0.05, 0.95, f'r={correlation:.3f}\nRMSE={rmse:.3f}\nn={len(mice_probs)}',
                   transform=ax.transAxes, fontsize=11, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # Set labels and title
        ax.set_xlabel('Mouse Switch Probability', fontsize=12)
        ax.set_ylabel('Model Switch Probability', fontsize=12)
        ax.set_title(model_type, fontsize=14)
        ax.grid(True, alpha=0.3)
    
    # Create legend for subjects
    if legend_handles:
        # Use more columns for legend (15-20 columns depending on number of subjects)
        n_subjects_in_legend = len(legend_labels)
        if n_subjects_in_legend <= 20:
            ncol = min(15, n_subjects_in_legend)
        elif n_subjects_in_legend <= 40:
            ncol = 20
        else:
            ncol = 25
        
        # Calculate how many rows the legend will need
        n_legend_rows = int(np.ceil(n_subjects_in_legend / ncol))
        
        # Adjust margins: more space at bottom for legend, less at top for title
        # Base bottom margin + additional space per legend row
        bottom_margin = 0.10 + (n_legend_rows - 1) * 0.035
        top_margin = 0.9  # Leave less space at top
        
        fig.legend(legend_handles, legend_labels, loc='lower center', 
                  bbox_to_anchor=(0.5, -0.015), 
                  ncol=ncol, fontsize=7, frameon=True, fancybox=True, shadow=True, 
                  title='Subject IDs', title_fontsize=9)
        
        plt.subplots_adjust(bottom=bottom_margin, top=top_margin)
    else:
        plt.subplots_adjust(top=0.9)

    # Add main title (using fig.text for better control)
    fig.text(0.5, 0.985, f'{task_type}: Pattern {pattern} ({n_back} trial{"s" if n_back > 1 else ""} back)',
             ha='center', fontsize=16, fontweight='bold')
    plt.show()


def plot_all_patterns_subject_level(subject_results, unique_subjects, task_type, 
                                    model_types, pattern_type='abstract', min_trials=5):
    """
    Create individual subject-level comparison figures for all history patterns.
    
    Parameters:
    -----------
    subject_results : dict
        Results from compute_subject_level_switch_probabilities
    unique_subjects : array-like
        Array of unique subject IDs
    task_type : str
        Task type being analyzed
    model_types : list
        List of model types to compare
    pattern_type : str
        'detailed' or 'abstract' patterns
    min_trials : int
        Minimum number of trials required to include a pattern
    """
    # Collect all unique patterns that have sufficient data
    all_patterns_by_nback = {}
    
    for n_back in [1, 2, 3]:
        patterns_with_data = set()
        
        for subject_id in unique_subjects:
            if subject_id not in subject_results['mice']:
                continue
            
            mice_patterns = subject_results['mice'][subject_id][pattern_type][n_back]
            
            for pattern in mice_patterns.keys():
                if mice_patterns[pattern]['total'] >= min_trials:
                    # Check if at least one model has this pattern for this subject
                    for model_type in model_types:
                        if subject_id in subject_results[model_type]:
                            model_patterns = subject_results[model_type][subject_id][pattern_type][n_back]
                            if (pattern in model_patterns and 
                                model_patterns[pattern]['total'] >= min_trials and
                                not np.isnan(model_patterns[pattern]['switch_probability'])):
                                patterns_with_data.add(pattern)
                                break
        
        all_patterns_by_nback[n_back] = sorted(list(patterns_with_data))
    
    # Print summary
    print(f"\n{'='*60}")
    print(f"Generating subject-level pattern comparison figures for {task_type}")
    print(f"Pattern type: {pattern_type}")
    print(f"{'='*60}")
    
    for n_back in [1, 2, 3]:
        print(f"\n{n_back} trial{'s' if n_back > 1 else ''} back: {len(all_patterns_by_nback[n_back])} patterns")
        if all_patterns_by_nback[n_back]:
            print(f"  Patterns: {', '.join(all_patterns_by_nback[n_back])}")
    
    # Create a figure for each pattern
    total_patterns = sum(len(patterns) for patterns in all_patterns_by_nback.values())
    pattern_count = 0
    
    for n_back in [1, 2, 3]:
        for pattern in all_patterns_by_nback[n_back]:
            pattern_count += 1
            print(f"\nPlotting pattern {pattern_count}/{total_patterns}: {pattern}")
            
            plot_single_pattern_subject_level(
                subject_results, unique_subjects, task_type, model_types, pattern,
                pattern_type=pattern_type, min_trials=min_trials
            )


def analyze_all_tasks_subject_level(df_session_history, dict_simulation_results,
                                    task_types, model_types, later_stages,
                                    pattern_type='abstract', min_trials=5):
    """
    Perform subject-level analysis for all task types.
    
    Parameters:
    -----------
    df_session_history : pd.DataFrame
        Real mice session data
    dict_simulation_results : dict
        Dictionary of simulation results
    task_types : list
        List of task types to analyze
    model_types : list
        List of model types to analyze
    later_stages : list
        List of later stage names to filter for
    pattern_type : str
        'detailed' or 'abstract' patterns
    min_trials : int
        Minimum number of trials required to include a subject-pattern combination
        
    Returns:
    --------
    all_subject_results : dict
        Dictionary containing subject-level results for all task types
    """
    all_subject_results = {}
    
    for task_type in task_types:
        print(f"\n{'='*80}")
        print(f"Processing subject-level analysis for {task_type}")
        print(f"{'='*80}")
        
        subject_results, unique_subjects = compute_subject_level_switch_probabilities(
            df_session_history, dict_simulation_results,
            task_type, model_types, later_stages,
            pattern_type=pattern_type, min_trials=min_trials
        )
        
        all_subject_results[task_type] = {
            'results': subject_results,
            'subjects': unique_subjects
        }
        
        plot_all_patterns_subject_level(
            subject_results, unique_subjects, task_type, model_types,
            pattern_type=pattern_type, min_trials=min_trials
        )
    
    return all_subject_results

In [ ]:
# Run subject-level analysis for all task types
task_types = [
    'Uncoupled Without Baiting', 
    'Uncoupled Baiting',
    'Coupled Baiting'
]

model_types = [
    'QLearning_L2F1_softmax', 
    'QLearning_L1F1_CK1_softmax', 
    'ForagingCompareThreshold', 
    # 'CTTAvg'
]

subject_level_results = analyze_all_tasks_subject_level(
    df_session_history, dict_simulation_results,
    task_types, model_types, later_stages,
    pattern_type='abstract',
    min_trials=5  # Adjust this threshold as needed
)

## analyze mice variability

In [ ]:
import random

def analyze_mice_data_variability(df_session_history, task_type, later_stages, 
                                 pattern_type='abstract', min_trials=10, 
                                 n_splits=1, random_seed=42):
    """
    Analyze variability in switch probabilities by randomly splitting mice data into two sets.
    
    Parameters:
    -----------
    df_session_history : pd.DataFrame
        Real mice session data
    task_type : str
        Task type to analyze
    later_stages : list
        List of later stage names to filter for
    pattern_type : str
        'detailed' or 'abstract' patterns
    min_trials : int
        Minimum number of trials required to include a pattern
    n_splits : int
        Number of random splits to perform
    random_seed : int
        Random seed for reproducibility
        
    Returns:
    --------
    split_results : dict
        Dictionary containing results for each split
    """
    random.seed(random_seed)
    np.random.seed(random_seed)
    
    # Filter data for the specific task type and stages
    df_mice_task = df_session_history[
        (df_session_history['curriculum_name'] == task_type) &
        (df_session_history['current_stage_actual'].isin(later_stages))
    ].copy()
    
    print(f"Total sessions for {task_type}: {len(df_mice_task)}")
    
    # Get unique subjects
    unique_subjects = df_mice_task['subject_id'].unique()
    print(f"Total unique subjects: {len(unique_subjects)}")
    
    split_results = {}
    
    for split_idx in range(n_splits):
        print(f"\nProcessing split {split_idx + 1}/{n_splits}")
        
        # Randomly split subjects into two groups
        shuffled_subjects = unique_subjects.copy()
        np.random.shuffle(shuffled_subjects)
        
        mid_point = len(shuffled_subjects) // 2
        subjects_split1 = shuffled_subjects[:mid_point]
        subjects_split2 = shuffled_subjects[mid_point:]
        
        print(f"Split 1: {len(subjects_split1)} subjects")
        print(f"Split 2: {len(subjects_split2)} subjects")
        
        # Create data subsets for each split
        df_split1 = df_mice_task[df_mice_task['subject_id'].isin(subjects_split1)]
        df_split2 = df_mice_task[df_mice_task['subject_id'].isin(subjects_split2)]
        
        print(f"Split 1: {len(df_split1)} sessions")
        print(f"Split 2: {len(df_split2)} sessions")
        
        # Compute switch probabilities for each split
        results_split1 = compute_switch_probabilities_by_history_pattern(
            df_split1, max_trials_back=3
        )
        results_split2 = compute_switch_probabilities_by_history_pattern(
            df_split2, max_trials_back=3
        )
        
        split_results[f'split_{split_idx}'] = {
            'split1': results_split1,
            'split2': results_split2,
            'subjects_split1': subjects_split1,
            'subjects_split2': subjects_split2
        }
    
    return split_results


def plot_mice_split_comparison(split_results, task_type, split_idx=0, 
                              pattern_type='abstract', min_trials=10):
    """
    Create comparison plots between two splits of mice data.
    
    Parameters:
    -----------
    split_results : dict
        Results from analyze_mice_data_variability
    task_type : str
        Task type being analyzed
    split_idx : int
        Which split to plot (if multiple splits were generated)
    pattern_type : str
        'detailed' or 'abstract' patterns
    min_trials : int
        Minimum number of trials required to include a pattern in the plot
    """
    split_key = f'split_{split_idx}'
    results_split1 = split_results[split_key]['split1']
    results_split2 = split_results[split_key]['split2']
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6), dpi=300)
    
    # Create color palette for patterns
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    colors = np.vstack([
        colors,
        plt.cm.Set3(np.linspace(0, 1, 12)),
        plt.cm.Pastel1(np.linspace(0, 1, 9)),
        plt.cm.Dark2(np.linspace(0, 1, 8))
    ])
    
    # Collect all unique patterns across both splits and all n_back values
    all_patterns = set()
    for n_back in [1, 2, 3]:
        split1_patterns = results_split1[pattern_type][n_back]
        split2_patterns = results_split2[pattern_type][n_back]
        
        for pattern in split1_patterns.keys():
            if (pattern in split2_patterns and 
                split1_patterns[pattern]['total'] >= min_trials and 
                split2_patterns[pattern]['total'] >= min_trials and
                not np.isnan(split1_patterns[pattern]['switch_probability']) and
                not np.isnan(split2_patterns[pattern]['switch_probability'])):
                all_patterns.add(pattern)
    
    all_patterns = sorted(list(all_patterns))
    pattern_colors = {pattern: colors[i % len(colors)] for i, pattern in enumerate(all_patterns)}
    
    # Track legend handles and labels
    legend_handles = {}
    legend_labels = {}
    
    for n_back in [1, 2, 3]:
        ax = axes[n_back-1]
        
        split1_patterns = results_split1[pattern_type][n_back]
        split2_patterns = results_split2[pattern_type][n_back]
        
        # Find common patterns with sufficient data
        common_patterns = []
        for pattern in split1_patterns.keys():
            if (pattern in split2_patterns and 
                split1_patterns[pattern]['total'] >= min_trials and 
                split2_patterns[pattern]['total'] >= min_trials and
                not np.isnan(split1_patterns[pattern]['switch_probability']) and
                not np.isnan(split2_patterns[pattern]['switch_probability'])):
                common_patterns.append(pattern)
        
        if not common_patterns:
            ax.text(0.5, 0.5, 'No common patterns\nwith sufficient data', 
                   transform=ax.transAxes, ha='center', va='center', fontsize=12)
            ax.set_title(f'{n_back} trial{"s" if n_back > 1 else ""} back')
            continue
        
        # Extract data for plotting
        split1_probs = []
        split2_probs = []
        split1_sems = []
        split2_sems = []
        
        for pattern in common_patterns:
            split1_prob = split1_patterns[pattern]['switch_probability']
            split2_prob = split2_patterns[pattern]['switch_probability']
            split1_sem = split1_patterns[pattern]['switch_probability_sem']
            split2_sem = split2_patterns[pattern]['switch_probability_sem']
            
            split1_probs.append(split1_prob)
            split2_probs.append(split2_prob)
            split1_sems.append(split1_sem)
            split2_sems.append(split2_sem)
            
            # Plot point with error bars
            handle = ax.errorbar(split1_prob, split2_prob, 
                       xerr=split1_sem, yerr=split2_sem,
                       marker='o', markersize=8, 
                       color=pattern_colors[pattern],
                       capsize=3, alpha=0.8, linewidth=2)
            
            # Store legend information
            if pattern not in legend_handles:
                legend_handles[pattern] = handle
                legend_labels[pattern] = pattern
        
        # Add unity line
        if split1_probs and split2_probs:
            max_val = max(max(split1_probs), max(split2_probs))
            min_val = min(min(split1_probs), min(split2_probs))
            
            ax.plot([min_val, max_val], [min_val, max_val], 
                   'k--', alpha=0.5, linewidth=2)
            
            # Set equal aspect ratio and limits
            ax.set_aspect('equal')
            buffer = 0.05
            ax.set_xlim(min_val - buffer, max_val + buffer)
            ax.set_ylim(min_val - buffer, max_val + buffer)
        
        # Set labels and title
        ax.set_xlabel('Split 1 Switch Probability', fontsize=12)
        ax.set_ylabel('Split 2 Switch Probability', fontsize=12)
        ax.set_title(f'{n_back} trial{"s" if n_back > 1 else ""} back', fontsize=14)
        ax.grid(True, alpha=0.3)
    
    # Add main title
    n_subjects_1 = len(split_results[split_key]['subjects_split1'])
    n_subjects_2 = len(split_results[split_key]['subjects_split2'])
    plt.suptitle(f'Mice Data Variability: {task_type}\n{pattern_type.capitalize()} Patterns (Split 1: {n_subjects_1} subjects, Split 2: {n_subjects_2} subjects)', 
                 fontsize=16, y=1.0)
    
    # Create legend
    sorted_patterns = sorted(legend_handles.keys())
    handles = [legend_handles[pattern] for pattern in sorted_patterns]
    labels = [legend_labels[pattern] for pattern in sorted_patterns]
    
    n_patterns = len(sorted_patterns)
    if n_patterns <= 8:
        ncol = 4
    elif n_patterns <= 16:
        ncol = 8
    else:
        ncol = 16
    
    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.0), 
              ncol=ncol, fontsize=10, frameon=True, fancybox=True, shadow=True)
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15 if n_patterns > 16 else 0.1)
    plt.show()
    
    # Print correlation statistics
    print(f"\nCorrelation statistics for mice data splits - {task_type} - {pattern_type} patterns:")
    print("="*80)
    
    for n_back in [1, 2, 3]:
        split1_patterns = results_split1[pattern_type][n_back]
        split2_patterns = results_split2[pattern_type][n_back]
        
        # Find common patterns
        common_patterns = []
        split1_probs = []
        split2_probs = []
        
        for pattern in split1_patterns.keys():
            if (pattern in split2_patterns and 
                split1_patterns[pattern]['total'] >= min_trials and 
                split2_patterns[pattern]['total'] >= min_trials and
                not np.isnan(split1_patterns[pattern]['switch_probability']) and
                not np.isnan(split2_patterns[pattern]['switch_probability'])):
                
                common_patterns.append(pattern)
                split1_probs.append(split1_patterns[pattern]['switch_probability'])
                split2_probs.append(split2_patterns[pattern]['switch_probability'])
        
        if len(split1_probs) > 1:
            correlation = np.corrcoef(split1_probs, split2_probs)[0, 1]
            rmse = np.sqrt(np.mean((np.array(split1_probs) - np.array(split2_probs))**2))
            print(f"  {n_back} trials back: r={correlation:.3f}, RMSE={rmse:.3f}, n_patterns={len(common_patterns)}")
        else:
            print(f"  {n_back} trials back: insufficient data")


def analyze_all_task_variability(df_session_history, task_types, later_stages, 
                                pattern_type='abstract', min_trials=10, 
                                n_splits=1, random_seed=42):
    """
    Analyze variability for all task types.
    
    Parameters:
    -----------
    df_session_history : pd.DataFrame
        Real mice session data
    task_types : list
        List of task types to analyze
    later_stages : list
        List of later stage names to filter for
    pattern_type : str
        'detailed' or 'abstract' patterns
    min_trials : int
        Minimum number of trials required to include a pattern
    n_splits : int
        Number of random splits to perform
    random_seed : int
        Random seed for reproducibility
        
    Returns:
    --------
    all_variability_results : dict
        Dictionary containing variability results for all task types
    """
    all_variability_results = {}
    
    for task_type in task_types:
        print(f"\n{'='*50}")
        print(f"Analyzing variability for {task_type}")
        print(f"{'='*50}")
        
        split_results = analyze_mice_data_variability(
            df_session_history, task_type, later_stages,
            pattern_type=pattern_type, min_trials=min_trials,
            n_splits=n_splits, random_seed=random_seed
        )
        
        all_variability_results[task_type] = split_results
        
        # Plot results for the first split
        plot_mice_split_comparison(
            split_results, task_type, split_idx=0,
            pattern_type=pattern_type, min_trials=min_trials
        )
    
    return all_variability_results

In [ ]:
# Analyze variability for all task types
task_types = [
    'Uncoupled Without Baiting', 
    'Uncoupled Baiting',
    'Coupled Baiting'
]

variability_results = analyze_all_task_variability(
    df_session_history, task_types, later_stages,
    pattern_type='abstract',
    min_trials=10,
    n_splits=1,  # Can increase this to test multiple random splits
    random_seed=51
)